# 🔧 TestMate APR — v2
**Automated Program Repair using Qwen-Coder + LoRA**

Pipeline:
1. Install dependencies
2. Build enhanced dataset (63 bug patterns)
3. Fine-tune Qwen-Coder with QLoRA
4. Benchmark on 10 unseen QuixBugs
5. Export adapter

## 1️⃣ Install Dependencies

In [1]:
!pip uninstall -y transformers huggingface_hub accelerate peft bitsandbytes

Found existing installation: transformers 5.2.0
Uninstalling transformers-5.2.0:
  Successfully uninstalled transformers-5.2.0
Found existing installation: huggingface_hub 1.4.1
Uninstalling huggingface_hub-1.4.1:
  Successfully uninstalled huggingface_hub-1.4.1
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1


In [ ]:
%!pip install -q \
    transformers==4.38.2 \
    huggingface_hub==0.21.4 \
    accelerate==0.27.2 \
    peft==0.8.2 \
    bitsandbytes==0.42.0 \
    datasets tqdm scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 3.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 74.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.4/346.4 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 MB 17.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 94.6 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.

In [ ]:
import os, torch
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64'
os.environ['CUDA_HOME'] = '/usr/local/cuda'
!cp /usr/local/cuda/lib64/libcudart.so* /usr/lib/ 2>/dev/null || true
%!pip install -qU bitsandbytes --no-cache-dir

print(f'✅ PyTorch  : {torch.__version__}')
print(f'✅ CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'✅ GPU      : {torch.cuda.get_device_name(0)}')

from transformers import AutoTokenizer
print('✅ Transformers OK')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 242.4 MB/s eta 0:00:00a 0:00:01
✅ PyTorch  : 2.9.0+cu126
✅ CUDA     : True
✅ GPU      : Tesla T4
✅ Transformers OK


## 2️⃣ Configuration

In [4]:
# ── Paths ──────────────────────────────────────────────────────────────
MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"
CACHE_DIR   = "/kaggle/working/model_cache"
OUTPUT_DIR  = "/kaggle/working/qwen-testmate-adapter"
TRAIN_FILE  = "/kaggle/working/data/train_prepared.jsonl"
TEST_FILE   = "/kaggle/working/data/test_prepared.jsonl"

# ── Training hyperparams ───────────────────────────────────────────────
MAX_LENGTH        = 2048
BATCH_SIZE        = 1
GRAD_ACCUMULATION = 16
LEARNING_RATE     = 2e-4
NUM_EPOCHS        = 3

print('✅ Config ready')

✅ Config ready


## 3️⃣ Dataset Factory — Enhanced (63 bug patterns)

In [5]:
import json, random, os

# ══════════════════════════════════════════════════════════════════
# QuixBugs — 20 training bugs  (10 reserved for benchmark)
# ══════════════════════════════════════════════════════════════════
QUIXBUGS_TRAIN = {
    "bitcount": {
        "buggy": "def bitcount(n):\n    count = 0\n    while n:\n        n ^= n - 1\n        count += 1\n    return count",
        "fixed": "def bitcount(n):\n    count = 0\n    while n:\n        n &= n - 1\n        count += 1\n    return count",
        "issue": "Fix bitcount - XOR should be AND to correctly remove bits."
    },
    "quicksort": {
        "buggy": "def quicksort(arr):\n    if not arr: return []\n    pivot = arr[0]\n    lesser = quicksort([x for x in arr[1:] if x < pivot])\n    greater = quicksort([x for x in arr[1:] if x > pivot])\n    return lesser + [pivot] + greater",
        "fixed": "def quicksort(arr):\n    if not arr: return []\n    pivot = arr[0]\n    lesser = quicksort([x for x in arr[1:] if x < pivot])\n    greater = quicksort([x for x in arr[1:] if x >= pivot])\n    return lesser + [pivot] + greater",
        "issue": "Fix quicksort - greater partition must handle duplicates using >=."
    },
    "is_valid_parenthesization": {
        "buggy": "def is_valid_parenthesization(parens):\n    depth = 0\n    for p in parens:\n        if p == '(': depth += 1\n        else:\n            depth -= 1\n            if depth < 0: return False\n    return True",
        "fixed": "def is_valid_parenthesization(parens):\n    depth = 0\n    for p in parens:\n        if p == '(': depth += 1\n        else:\n            depth -= 1\n            if depth < 0: return False\n    return depth == 0",
        "issue": "Fix logic - final depth must be 0 for valid parenthesization."
    },
    "find_first_in_sorted": {
        "buggy": "def find_first_in_sorted(arr, x):\n    lo, hi = 0, len(arr)\n    while lo <= hi:\n        mid = (lo + hi) // 2\n        if x == arr[mid]: return mid\n        elif x < arr[mid]: hi = mid\n        else: lo = mid + 1\n    return -1",
        "fixed": "def find_first_in_sorted(arr, x):\n    lo, hi = 0, len(arr)\n    while lo < hi:\n        mid = (lo + hi) // 2\n        if x == arr[mid]: return mid\n        elif x < arr[mid]: hi = mid\n        else: lo = mid + 1\n    return -1",
        "issue": "Fix binary search - loop condition should be lo < hi."
    },
    "gcd": {
        "buggy": "def gcd(a, b):\n    if b == 0: return a\n    else: return gcd(a % b, b)",
        "fixed": "def gcd(a, b):\n    if b == 0: return a\n    else: return gcd(b, a % b)",
        "issue": "Fix GCD - swap arguments: gcd(b, a % b)."
    },
    "flatten": {
        "buggy": "def flatten(arr):\n    for x in arr:\n        if isinstance(x, list):\n            for y in flatten(x): yield y\n        else: yield flatten(x)",
        "fixed": "def flatten(arr):\n    for x in arr:\n        if isinstance(x, list):\n            for y in flatten(x): yield y\n        else: yield x",
        "issue": "Recursion error: yield flatten(x) used on non-list element."
    },
    "reverse_linked_list": {
        "buggy": "def reverse_linked_list(node):\n    prevnode = None\n    while node:\n        nextnode = node.next\n        node.next = prevnode\n        node = nextnode\n    return prevnode",
        "fixed": "def reverse_linked_list(node):\n    prevnode = None\n    while node:\n        nextnode = node.next\n        node.next = prevnode\n        prevnode = node\n        node = nextnode\n    return prevnode",
        "issue": "Pointer error: prevnode must be updated to current node."
    },
    "lcs": {
        "buggy": "def lcs(a, b):\n    if not a or not b: return ''\n    elif a[0] == b[0]: return a[0] + lcs(a[1:], b[1:])\n    else: return max(lcs(a, b[1:]), lcs(a[1:], b))",
        "fixed": "def lcs(a, b):\n    if not a or not b: return ''\n    elif a[0] == b[0]: return a[0] + lcs(a[1:], b[1:])\n    else: return max(lcs(a, b[1:]), lcs(a[1:], b), key=len)",
        "issue": "Logic error: max() needs key=len to compare string lengths."
    },
    "bucketsort": {
        "buggy": "def bucketsort(arr, k):\n    counts = [0] * k\n    for x in arr: counts[x] += 1\n    sorted_arr = []\n    for i, count in enumerate(arr):\n        sorted_arr.extend([i] * count)\n    return sorted_arr",
        "fixed": "def bucketsort(arr, k):\n    counts = [0] * k\n    for x in arr: counts[x] += 1\n    sorted_arr = []\n    for i, count in enumerate(counts):\n        sorted_arr.extend([i] * count)\n    return sorted_arr",
        "issue": "Loop error: iterate over 'counts' not 'arr'."
    },
    "hanoi": {
        "buggy": "def hanoi(height, start=1, end=3):\n    steps = []\n    if height > 0:\n        helper = ({1, 2, 3} - {start} - {end}).pop()\n        steps.extend(hanoi(height - 1, start, helper))\n        steps.append((start, helper))\n        steps.extend(hanoi(height - 1, helper, end))\n    return steps",
        "fixed": "def hanoi(height, start=1, end=3):\n    steps = []\n    if height > 0:\n        helper = ({1, 2, 3} - {start} - {end}).pop()\n        steps.extend(hanoi(height - 1, start, helper))\n        steps.append((start, end))\n        steps.extend(hanoi(height - 1, helper, end))\n    return steps",
        "issue": "Logic error: disc must move to 'end' peg not 'helper'."
    },
    "pascal": {
        "buggy": "def pascal(n):\n    rows = [[1]]\n    for r in range(1, n):\n        row = []\n        for c in range(0, r):\n            v = (rows[r-1][c-1] if c > 0 else 0) + (rows[r-1][c] if c < r else 0)\n            row.append(v)\n        rows.append(row)\n    return rows",
        "fixed": "def pascal(n):\n    rows = [[1]]\n    for r in range(1, n):\n        row = []\n        for c in range(0, r + 1):\n            v = (rows[r-1][c-1] if c > 0 else 0) + (rows[r-1][c] if c < r else 0)\n            row.append(v)\n        rows.append(row)\n    return rows",
        "issue": "Off-by-one: column loop must go to r + 1."
    },
    "rpn_eval": {
        "buggy": "def rpn_eval(tokens):\n    def op(s, a, b): return {'+':a+b, '-':a-b, '*':a*b, '/':a/b}[s]\n    stack = []\n    for t in tokens:\n        if isinstance(t, float): stack.append(t)\n        else:\n            a = stack.pop(); b = stack.pop()\n            stack.append(op(t, a, b))\n    return stack.pop()",
        "fixed": "def rpn_eval(tokens):\n    def op(s, a, b): return {'+':a+b, '-':a-b, '*':a*b, '/':a/b}[s]\n    stack = []\n    for t in tokens:\n        if isinstance(t, float): stack.append(t)\n        else:\n            b = stack.pop(); a = stack.pop()\n            stack.append(op(t, a, b))\n    return stack.pop()",
        "issue": "Operand order: first pop is 'b', second is 'a'."
    },
    "topological_sort": {
        "buggy": "def topological_sort(nodes):\n    ordered_nodes = []\n    for node in nodes:\n        if not node.predecessors: ordered_nodes.append(node)\n    for node in ordered_nodes:\n        for successor in node.successors:\n            if successor not in ordered_nodes:\n                ordered_nodes.append(successor)\n    return ordered_nodes",
        "fixed": "def topological_sort(nodes):\n    ordered_nodes = []\n    for node in nodes:\n        if not node.predecessors: ordered_nodes.append(node)\n    for node in ordered_nodes:\n        for s in node.successors:\n            if all(p in ordered_nodes for p in s.predecessors) and s not in ordered_nodes:\n                ordered_nodes.append(s)\n    return ordered_nodes",
        "issue": "Logic: successor must wait for all predecessors."
    },
    "wrap": {
        "buggy": "def wrap(text, cols):\n    lines = []\n    while len(text) > cols:\n        end = text.rfind(' ', 0, cols)\n        if end == -1: end = cols\n        lines.append(text[:end])\n        text = text[end:]\n    lines.append(text)\n    return lines",
        "fixed": "def wrap(text, cols):\n    lines = []\n    while len(text) > cols:\n        end = text.rfind(' ', 0, cols)\n        if end == -1: end = cols\n        lines.append(text[:end])\n        text = text[end:].strip()\n    lines.append(text)\n    return lines",
        "issue": "Missing strip() to remove leading spaces."
    },
    "next_permutation": {
        "buggy": "def next_permutation(arr):\n    for i in range(len(arr) - 2, -1, -1):\n        if arr[i] < arr[i + 1]:\n            for j in range(len(arr) - 1, i, -1):\n                if arr[j] < arr[i]:\n                    arr[i], arr[j] = arr[j], arr[i]\n                    arr[i + 1 :] = reversed(arr[i + 1 :])\n                    return arr\n    return None",
        "fixed": "def next_permutation(arr):\n    for i in range(len(arr) - 2, -1, -1):\n        if arr[i] < arr[i + 1]:\n            for j in range(len(arr) - 1, i, -1):\n                if arr[j] > arr[i]:\n                    arr[i], arr[j] = arr[j], arr[i]\n                    arr[i + 1 :] = reversed(arr[i + 1 :])\n                    return arr\n    return None",
        "issue": "Swap condition should be arr[j] > arr[i]."
    },
    "kth": {
        "buggy": "def kth(arr, k):\n    pivot = arr[0]\n    below = [x for x in arr if x < pivot]\n    above = [x for x in arr if x > pivot]\n    num_less = len(below)\n    num_less_or_equal = len(arr) - len(above)\n    if k < num_less: return kth(below, k)\n    elif k >= num_less_or_equal: return kth(above, k - num_less_or_equal)\n    else: return pivot",
        "fixed": "def kth(arr, k):\n    pivot = arr[0]\n    below = [x for x in arr[1:] if x < pivot]\n    above = [x for x in arr[1:] if x >= pivot]\n    num_less = len(below)\n    if k < num_less: return kth(below, k)\n    elif k == num_less: return pivot\n    else: return kth(above, k - num_less - 1)",
        "issue": "Pivot must be excluded from recursive slices."
    },
    "shunting_yard": {
        "buggy": "def shunting_yard(tokens):\n    precedences = {'+': 1, '-': 1, '*': 2, '/': 2}\n    rpndict = []\n    opstack = []\n    for token in tokens:\n        if isinstance(token, int): rpndict.append(token)\n        else:\n            while opstack and precedences[token] <= precedences[opstack[-1]]:\n                rpndict.append(opstack.pop())\n            opstack.append(token)\n    while opstack: rpndict.append(opstack.pop())\n    return rpndict",
        "fixed": "def shunting_yard(tokens):\n    precedences = {'+': 1, '-': 1, '*': 2, '/': 2}\n    rpndict = []\n    opstack = []\n    for token in tokens:\n        if isinstance(token, int): rpndict.append(token)\n        else:\n            while opstack and precedences[token] < precedences[opstack[-1]]:\n                rpndict.append(opstack.pop())\n            opstack.append(token)\n    while opstack: rpndict.append(opstack.pop())\n    return rpndict",
        "issue": "Precedence comparison should be < not <=."
    },
    "subsequences": {
        "buggy": "def subsequences(a, b, k):\n    if k == 0: return [[]]\n    ret = []\n    for i in range(a, b + 1 - k):\n        ret.extend([i] + rest for rest in subsequences(i + 1, b, k - 1))\n    return ret",
        "fixed": "def subsequences(a, b, k):\n    if k == 0: return [[]]\n    ret = []\n    for i in range(a, b + 1 - k + 1):\n        ret.extend([[i] + rest for rest in subsequences(i + 1, b, k - 1)])\n    return ret",
        "issue": "Off-by-one: loop range must include b + 1 - k."
    },
    "possible_change": {
        "buggy": "def possible_change(coins, amount):\n    if amount == 0: return 1\n    if not coins: return 0\n    c, *rest = coins\n    return possible_change(coins, amount - c) + possible_change(rest, amount)",
        "fixed": "def possible_change(coins, amount):\n    if amount == 0: return 1\n    if amount < 0 or not coins: return 0\n    c, *rest = coins\n    return possible_change(coins, amount - c) + possible_change(rest, amount)",
        "issue": "Missing base case: handle negative amount."
    },
    "shortest_path_length": {
        "buggy": "def shortest_path_length(startnode, goalnode):\n    unvisited_nodes = []\n    dist = {startnode: 0}\n    while unvisited_nodes:\n        current_node = min(unvisited_nodes, key=lambda node: dist.get(node, float('inf')))\n        unvisited_nodes.remove(current_node)\n        if current_node is goalnode: return dist[current_node]\n        for nextnode, distance in current_node.successors.items():\n            new_dist = dist[current_node] + distance\n            if new_dist < dist.get(nextnode, float('inf')):\n                dist[nextnode] = new_dist\n    return float('inf')",
        "fixed": "def shortest_path_length(startnode, goalnode):\n    from heapq import heappush, heappop\n    queue = [(0, startnode)]\n    visited = set()\n    while queue:\n        d, node = heappop(queue)\n        if node in visited: continue\n        visited.add(node)\n        if node is goalnode: return d\n        for nextnode, dist in node.successors.items():\n            heappush(queue, (d + dist, nextnode))\n    return float('inf')",
        "issue": "Dijkstra needs a priority queue."
    },
}
print(f'✅ QuixBugs loaded: {len(QUIXBUGS_TRAIN)} training bugs')

✅ QuixBugs loaded: 20 training bugs


In [6]:
# ══════════════════════════════════════════════════════════════════
# 63 Synthetic Bug Patterns across 9 categories
# (func_name, params, buggy_body, fixed_body, issue, error_type, category)
# ══════════════════════════════════════════════════════════════════
SYNTHETIC_PATTERNS = [
    # Off-by-one
    ("check_limit",   "v, l",    "return v > l",                         "return v >= l",                           "Off-by-one: inclusive upper limit requires >=",            "AssertionError",    "off_by_one"),
    ("check_lower",   "v, l",    "return v < l",                         "return v <= l",                           "Off-by-one: inclusive lower limit requires <=",            "AssertionError",    "off_by_one"),
    ("index_last",    "arr",     "return arr[len(arr)]",                  "return arr[len(arr) - 1]",                "Index error: last element is at len-1",                   "IndexError",        "off_by_one"),
    ("slice_head",    "arr, k",  "return arr[:k-1]",                     "return arr[:k]",                          "Slice error: arr[:k] correctly includes index k-1",       "AssertionError",    "off_by_one"),
    ("count_items",   "arr",     "return len(arr) - 1",                  "return len(arr)",                         "Off-by-one: length must not be decremented",              "AssertionError",    "off_by_one"),
    ("mid_index",     "lo, hi",  "return (lo + hi) / 2",                 "return (lo + hi) // 2",                   "Type error: integer division needed for index",           "TypeError",         "off_by_one"),
    ("loop_n",        "n",       "return list(range(1, n))",             "return list(range(n))",                   "Off-by-one: range(n) gives n items, range(1,n) gives n-1","AssertionError",    "off_by_one"),
    # Logic
    ("verify_id",     "uid",     "return uid is 10",                     "return uid == 10",                        "Identity vs Equality: use == not 'is' for integers",      "AssertionError",    "logic"),
    ("is_even",       "n",       "return n % 2 == 1",                    "return n % 2 == 0",                       "Logic error: even numbers have remainder 0",               "AssertionError",    "logic"),
    ("is_positive",   "n",       "return n > 1",                         "return n > 0",                            "Logic error: positive includes 1, use > 0",               "AssertionError",    "logic"),
    ("all_true",      "flags",   "return any(flags)",                    "return all(flags)",                       "Logic: all() requires every flag to be True",             "AssertionError",    "logic"),
    ("in_range",      "x, a, b", "return a < x < b",                    "return a <= x <= b",                      "Logic: inclusive range check requires <=",                "AssertionError",    "logic"),
    ("toggle",        "flag",    "return flag",                          "return not flag",                         "Logic error: toggle must negate the boolean",             "AssertionError",    "logic"),
    ("safe_div",      "a, b",    "return a / b",                         "return a / b if b != 0 else 0",           "ZeroDivisionError: guard against zero denominator",       "ZeroDivisionError", "logic"),
    ("negate",        "n",       "return n * -0",                        "return n * -1",                           "Logic error: -0 is 0, use -1 to negate",                  "AssertionError",    "logic"),
    ("abs_val",       "n",       "return n if n > 0 else n",             "return n if n >= 0 else -n",              "Logic: absolute value must negate negative numbers",      "AssertionError",    "logic"),
    ("clamp",         "x, lo, hi","return max(lo, x)",                  "return max(lo, min(x, hi))",              "Logic: clamp must also apply upper bound",                "AssertionError",    "logic"),
    ("xor_check",     "a, b",    "return a and b",                       "return bool(a) ^ bool(b)",                "Logic: XOR differs from AND",                             "AssertionError",    "logic"),
    # Collection
    ("is_active",     "data",    "return len(data) > 0",                 "return bool(data)",                       "Logic: use implicit boolean for non-empty sequences",     "AssertionError",    "collection"),
    ("first_item",    "arr",     "return arr[1]",                        "return arr[0]",                           "Index error: first item is at index 0",                   "IndexError",        "collection"),
    ("safe_get",      "d, k",    "return d[k]",                          "return d.get(k)",                         "KeyError: use dict.get() for safe access",                "KeyError",          "collection"),
    ("append_extend", "lst, items","lst.append(items)\n    return lst",  "lst.extend(items)\n    return lst",       "Logic: append adds one object, extend adds elements",     "AssertionError",    "collection"),
    ("remove_dup",    "arr",     "return list(arr)",                     "return list(set(arr))",                   "Logic: deduplication requires converting to set",         "AssertionError",    "collection"),
    ("dict_assign",   "d, k, v", "d[k] == v\n    return d",             "d[k] = v\n    return d",                  "Assignment error: == compares, = assigns",                "AssertionError",    "collection"),
    ("stack_top",     "stack",   "return stack[0]",                      "return stack[-1]",                        "Stack: LIFO means last element is at index -1",           "AssertionError",    "collection"),
    ("count_key",     "d, k",    "return d[k]",                          "return d.get(k, 0)",                      "KeyError: missing key should default to 0",               "KeyError",          "collection"),
    ("flatten_one",   "lst",     "return [x for x in lst]",             "return [x for sub in lst for x in sub]",  "Logic: nested list needs double iteration",               "AssertionError",    "collection"),
    # String
    ("is_empty_str",  "s",       "return s == None",                     "return s == ''",                          "Type error: empty string is '' not None",                 "AssertionError",    "string"),
    ("trim",          "s",       "return s.strip",                       "return s.strip()",                        "Call error: strip is a method, must use ()",             "TypeError",         "string"),
    ("starts_with",   "s, prefix","return prefix in s",                 "return s.startswith(prefix)",             "Logic: 'in' checks anywhere, startswith checks start",   "AssertionError",    "string"),
    ("reverse_str",   "s",       "return s.reverse()",                   "return s[::-1]",                          "AttributeError: strings have no .reverse()",             "AttributeError",    "string"),
    ("char_at",       "s, i",    "return s[i:]",                         "return s[i]",                             "Slice vs index: s[i] gets character",                     "AssertionError",    "string"),
    ("upper_call",    "s",       "return s.upper",                       "return s.upper()",                        "Call error: upper() needs parentheses",                   "TypeError",         "string"),
    ("repeat_str",    "s, n",    "return s * 0",                         "return s * n",                            "Logic: multiplier must be n not 0",                       "AssertionError",    "string"),
    ("str_contains",  "s, sub",  "return s == sub",                      "return sub in s",                         "Logic: == checks equality, 'in' checks containment",     "AssertionError",    "string"),
    # Recursion
    ("factorial",     "n",       "if n == 0: return 0\n    return n * factorial(n - 1)",   "if n == 0: return 1\n    return n * factorial(n - 1)",   "Base case: factorial(0) must return 1",    "AssertionError", "recursion"),
    ("fib",           "n",       "if n <= 1: return 0\n    return fib(n-1) + fib(n-2)",   "if n <= 1: return n\n    return fib(n-1) + fib(n-2)",   "Base case: fib(1) must return 1",          "AssertionError", "recursion"),
    ("power",         "base, exp","if exp == 0: return 0\n    return base * power(base, exp-1)", "if exp == 0: return 1\n    return base * power(base, exp-1)", "Base case: base^0 is 1 not 0",    "AssertionError", "recursion"),
    ("depth",         "node",    "if not node: return 1\n    return 1 + max(depth(node.left), depth(node.right))", "if not node: return 0\n    return 1 + max(depth(node.left), depth(node.right))", "Base case: depth of empty node is 0", "AssertionError","recursion"),
    ("count_nodes",   "node",    "if not node: return 1\n    return 1 + count_nodes(node.left) + count_nodes(node.right)", "if not node: return 0\n    return 1 + count_nodes(node.left) + count_nodes(node.right)", "Base case: empty node count is 0", "AssertionError","recursion"),
    # Type
    ("parse_int",     "s",       "return float(s)",                      "return int(s)",                           "Type error: integer parsing needs int() not float()",     "AssertionError",    "type"),
    ("to_bool",       "val",     "return val == True",                   "return bool(val)",                        "Type: bool() handles truthy values",                      "AssertionError",    "type"),
    ("none_check",    "val",     "return val == None",                   "return val is None",                      "Use 'is None' not '== None'",                             "AssertionError",    "type"),
    ("int_div",       "a, b",    "return a / b",                         "return a // b",                           "Type: integer division requires //",                      "AssertionError",    "type"),
    ("cast_round",    "x",       "return int(x)",                        "return round(x)",                         "Logic: round() for nearest integer",                      "AssertionError",    "type"),
    ("to_str",        "n",       "return n",                             "return str(n)",                           "Type: must convert number to string",                     "TypeError",         "type"),
    # Loop
    ("find_max",      "arr",     "best = 0\n    for x in arr:\n        if x > best: best = x\n    return best",   "best = arr[0]\n    for x in arr:\n        if x > best: best = x\n    return best",  "Init: use arr[0] not 0",    "AssertionError","loop"),
    ("product",       "arr",     "result = 0\n    for x in arr:\n        result *= x\n    return result",         "result = 1\n    for x in arr:\n        result *= x\n    return result",             "Init: product identity is 1","AssertionError","loop"),
    ("collect_evens", "arr",     "return [x for x in arr if x % 2 == 1]",               "return [x for x in arr if x % 2 == 0]",               "Filter: even numbers have remainder 0",    "AssertionError","loop"),
    ("running_total", "arr",     "out = []\n    s = 0\n    for x in arr:\n        out.append(x)\n        s += x\n    return out",  "out = []\n    s = 0\n    for x in arr:\n        s += x\n        out.append(s)\n    return out",  "Order: accumulate before appending","AssertionError","loop"),
    ("sum_squares",   "n",       "total = 0\n    for i in range(n):\n        total += i\n    return total",       "total = 0\n    for i in range(n):\n        total += i * i\n    return total",       "Logic: accumulate i*i not i",  "AssertionError","loop"),
    ("reverse_list",  "arr",     "return arr.sort()",                    "return arr[::-1]",                        "sort() sorts in-place and returns None",                  "AssertionError",    "loop"),
    # Guard
    ("safe_len",      "s",       "return len(s)",                        "return len(s) if s else 0",               "Guard: handle None/empty before len()",                   "TypeError",         "guard"),
    ("first_or_none", "arr",     "return arr[0]",                        "return arr[0] if arr else None",          "Guard: return None for empty list",                       "IndexError",        "guard"),
    ("max_or_zero",   "arr",     "return max(arr)",                      "return max(arr) if arr else 0",           "Guard: max() raises on empty sequence",                   "ValueError",        "guard"),
    ("safe_sqrt",     "x",       "import math\n    return math.sqrt(x)", "import math\n    return math.sqrt(x) if x >= 0 else 0", "Guard: sqrt undefined for negatives",    "ValueError",   "guard"),
    # Scope / mutation
    ("copy_list",     "lst",     "new = lst\n    return new",            "new = lst.copy()\n    return new",        "Reference: assignment copies reference not values",       "AssertionError",    "scope"),
    ("swap_vals",     "a, b",    "a = b\n    b = a\n    return a, b",    "a, b = b, a\n    return a, b",           "Swap: use tuple unpacking",                               "AssertionError",    "scope"),
    ("clear_list",    "lst",     "lst = []\n    return lst",             "lst.clear()\n    return lst",             "Scope: rebinding local name doesn't clear original",      "AssertionError",    "scope"),
]

print(f'✅ Synthetic patterns loaded: {len(SYNTHETIC_PATTERNS)}')
categories = set(p[6] for p in SYNTHETIC_PATTERNS)
print(f'   Categories: {categories}')

✅ Synthetic patterns loaded: 59
   Categories: {'string', 'loop', 'logic', 'type', 'collection', 'guard', 'recursion', 'scope', 'off_by_one'}


In [7]:
# ══════════════════════════════════════════════════════════════════
# Instruction templates + Traceback generator + Dataset builder
# ══════════════════════════════════════════════════════════════════
INSTRUCTION_TEMPLATES = [
    "Expert APR agent. Fix code using stack trace.",
    "You are an automated program repair system. Analyse the error and return only the corrected function.",
    "Repair the following buggy Python function. Output the fixed code only.",
    "Identify and fix the bug in the code below. Use the provided trace as a guide.",
    "You are a senior Python engineer. Fix the logic error and return the corrected function.",
]

def generate_traceback(error_type, func_name, line_no, detail=""):
    trace = "Traceback (most recent call last):\n"
    trace += f'  File "app/logic.py", line {line_no}, in {func_name}\n'
    msgs = {
        "ZeroDivisionError": "    return a / b\nZeroDivisionError: division by zero",
        "IndexError":        "    return arr[idx]\nIndexError: list index out of range",
        "KeyError":          "    return d[k]\nKeyError: key not found",
        "AttributeError":    "    return obj.method()\nAttributeError: object has no such attribute",
        "ValueError":        "    return math.sqrt(x)\nValueError: math domain error",
        "TypeError":         f"    result = func(arg)\nTypeError: {detail}",
    }
    trace += msgs.get(error_type, f"    assert output == expected\nAssertionError: Logic verification failed. {detail}")
    return trace

def build_and_save_datasets(quixbugs_repeats=200, total_synthetic=8000):
    quixbugs_data, synthetic_data = [], []

    for name, data in QUIXBUGS_TRAIN.items():
        for _ in range(quixbugs_repeats):
            quixbugs_data.append({
                "instruction": random.choice(INSTRUCTION_TEMPLATES),
                "input":  f"ISSUE: {data['issue']}\n\nCODE:\n{data['buggy']}",
                "output": data['fixed']
            })

    for _ in range(total_synthetic):
        f_name, params, buggy_body, fixed_body, issue, error_type, _ = random.choice(SYNTHETIC_PATTERNS)
        line_no    = random.randint(5, 60)
        trace      = generate_traceback(error_type, f_name, line_no, issue)
        buggy_code = f"def {f_name}({params}):\n" + "\n".join(f"    {l}" for l in buggy_body.split("\n"))
        fixed_code = f"def {f_name}({params}):\n" + "\n".join(f"    {l}" for l in fixed_body.split("\n"))
        synthetic_data.append({
            "instruction": random.choice(INSTRUCTION_TEMPLATES),
            "input":  f"ISSUE: {issue}\n\nTRACE:\n{trace}\n\nBUGGY:\n{buggy_code}",
            "output": fixed_code
        })

    random.shuffle(synthetic_data)
    test_data  = synthetic_data[-300:]        # test = synthetic only (no QuixBug leakage)
    train_data = quixbugs_data + synthetic_data[:-300]
    random.shuffle(train_data)

    def save_jsonl(data, filename):
        os.makedirs(os.path.dirname(filename), exist_ok=True)
        with open(filename, 'w', encoding='utf-8') as f:
            for item in data:
                prompt = (
                    f"<|im_start|>system\n{item['instruction']}<|im_end|>\n"
                    f"<|im_start|>user\n{item['input']}<|im_end|>\n"
                    f"<|im_start|>assistant\n{item['output']}<|im_end|>"
                )
                f.write(json.dumps({"text": prompt}, ensure_ascii=False) + '\n')

    save_jsonl(train_data, TRAIN_FILE)
    save_jsonl(test_data,  TEST_FILE)
    print(f"✅ Dataset ready!")
    print(f"   QuixBugs : {len(quixbugs_data):,} samples ({quixbugs_repeats}x augmentation)")
    print(f"   Synthetic: {total_synthetic:,} samples ({len(SYNTHETIC_PATTERNS)} patterns)")
    print(f"   Train    : {len(train_data):,}")
    print(f"   Test     : {len(test_data):,}")

build_and_save_datasets()

✅ Dataset ready!
   QuixBugs : 4,000 samples (200x augmentation)
   Synthetic: 8,000 samples (59 patterns)
   Train    : 11,700
   Test     : 300


## 4️⃣ Fine-Tune with QLoRA

In [8]:
import torch, numpy as np
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, TrainingArguments,
    Trainer, DataCollatorForLanguageModeling, BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset

# Fix numpy serialization
torch.serialization.add_safe_globals([
    np.ndarray, np._core.multiarray._reconstruct, np.dtype,
    np.dtypes.UInt32DType, np.random._pickle.__generator_ctor,
    np.random._pickle.__bit_generator_ctor
])

print('✅ Imports OK')

2026-02-23 20:56:36.745635: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771880196.964872      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771880197.027879      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771880197.512568      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771880197.512624      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771880197.512627      55 computation_placer.cc:177] computation placer alr

✅ Imports OK


In [9]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, cache_dir=CACHE_DIR)
tokenizer.pad_token = tokenizer.eos_token
print('✅ Tokenizer loaded')

# Tokenize dataset
raw_ds = load_dataset('json', data_files={'train': TRAIN_FILE, 'test': TEST_FILE})

def tokenize_fn(examples):
    return tokenizer(examples['text'], truncation=True, max_length=MAX_LENGTH, padding=False)

tokenized_ds = raw_ds.map(tokenize_fn, batched=True, remove_columns=raw_ds['train'].column_names)
print(f'✅ Dataset tokenized — train: {len(tokenized_ds["train"]):,} | test: {len(tokenized_ds["test"]):,}')

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


...

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/11700 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

✅ Dataset tokenized — train: 11,700 | test: 300


In [10]:
# Load model in 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4'
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    cache_dir=CACHE_DIR
)
model = prepare_model_for_kbit_training(model)
print('✅ Base model loaded in 4-bit')

# LoRA — includes MLP layers for better code reasoning
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    task_type='CAUSAL_LM'
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.33G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Base model loaded in 4-bit
trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273020662807087


In [11]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    fp16=True,
    gradient_checkpointing=True,      
    optim='paged_adamw_8bit',         
    save_steps=100,
    logging_steps=100,
    evaluation_strategy='steps',
    save_strategy='steps',
    load_best_model_at_end=True,
    report_to='none'
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['test'],
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
)

trainer.train()
print('✅ Training complete!')

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
100,0.288300,0.089560
200,0.073000,0.086987
300,0.069700,0.087462
400,0.070500,0.086958
500,0.070000,0.084488
600,0.070600,0.085388
700,0.069000,0.085656
800,0.069300,0.085957
900,0.070400,0.084987
1000,0.068300,0.085342


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

✅ Training complete!


In [12]:
# Save final adapter
import os
os.makedirs(f'{OUTPUT_DIR}/final', exist_ok=True)
model.save_pretrained(f'{OUTPUT_DIR}/final')
tokenizer.save_pretrained(f'{OUTPUT_DIR}/final')
print(f'✅ Adapter saved to {OUTPUT_DIR}/final')

✅ Adapter saved to /kaggle/working/qwen-testmate-adapter/final
